# Lesson 17 Lab — ModelOpt to TensorRT-LLM Quantization Pipelines

**Puzzle:** Which evidence is lost when a quantized checkpoint is handed from one tool to another?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Quantization pipelines cross tool boundaries: calibration may happen in ModelOpt, checkpoint export in one schema, and engine build in TensorRT-LLM. If model revision, recipe, scales, build flags, and rollback identity are not carried together, a fast engine cannot be reproduced or safely compared with its baseline.


## 0. Predict before running

1. List the fields required to reproduce a quantized checkpoint-to-engine handoff.
2. Explain why a scale checksum is useful but insufficient for engine identity.
3. Predict the decision when neither ModelOpt nor TensorRT-LLM is installed.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A ModelOpt-to-TensorRT-LLM handoff includes base revision, calibration corpus, recipe, per-layer exclusions, quantized tensor metadata, tokenizer, builder/runtime versions, engine flags, and rollback target.

- A pipeline needs immutable model revision, calibration recipe, quantization metadata, build flags, and engine identity.
- FP8, INT4, and FP4 are different recipes, not interchangeable compression levels.
- Package availability is only the first compatibility gate.


## 2. Derive the mechanism

Model optimization chooses and serializes a numerical representation; the engine builder lowers it to hardware tactics. Losing group axes, scale dtype, or recipe version at the boundary can change semantics even when files load.

A pipeline artifact is a directed chain: base model revision → calibration sample manifest → quantization recipe and scales → exported checkpoint → builder version/flags → engine → quality and performance report. Hashes establish byte identity at a boundary; semantic fields establish how those bytes should be interpreted.

FP8, INT4, and FP4 are different graph and scaling recipes, not points on one interchangeable slider. The manifest should therefore make format, group/block size, calibration, handoff status, and rollback target explicit. Missing stages remain false rather than being inferred from a numerical probe.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "17-modelopt-tensorrt-llm"
device = require_cuda()
torch.manual_seed(2026 + 17)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | versioned BF16 rollback revision |
| Candidate | INT4 handoff manifest with scale fingerprint |
| Held constant | fixed synthetic scale tensor, schema requirements, base/rollback identifiers |
| Measurements | manifest completeness, SHA-256 fingerprint, package availability, numerical Q/DQ error |
| Evidence | `compatibility-probe` |

**Experiment:** Generate and validate a quantization handoff manifest seeded by a CUDA numerical probe, while checking ModelOpt and TensorRT-LLM availability independently.


## 5. Read the experiment code

The notebook creates a complete handoff manifest and a CUDA numerical fingerprint while explicitly marking ModelOpt and TensorRT-LLM availability.

The notebook generates a small CUDA quantization fingerprint, hashes the scale bytes, and builds a manifest with required fields. It independently probes ModelOpt and TensorRT-LLM and records both handoff flags. Validation checks schema completeness, not engine success.

This is intentionally a pipeline-contract lab. The synthetic Q/DQ error catches accidental recipe changes, while the hash catches byte changes; neither substitutes for loading the exported checkpoint or building an engine.

Only after these variables match the protocol should the cell be executed.


In [2]:
import importlib.util, hashlib
w=torch.randn(256,256,device=device); _,scales,dq=symmetric_quantize(w,bits=4,group_size=64)
manifest={"base_revision":"example-frozen-revision","recipe":{"format":"INT4","group_size":64,"calibration":"synthetic-v1"},
          "handoff":{"modelopt":importlib.util.find_spec("modelopt") is not None,"tensorrt_llm":importlib.util.find_spec("tensorrt_llm") is not None},
          "scale_sha256":hashlib.sha256(scales.cpu().numpy().tobytes()).hexdigest(),"rollback_revision":"bf16-baseline-v1"}
required=("base_revision","recipe","handoff","scale_sha256","rollback_revision")
result=base_result(17,"compatibility-probe"); result.update({"manifest":manifest,"manifest_complete":all(k in manifest for k in required),
    "numerical_probe":error_metrics(w,dq),"conclusion":"The handoff contract was validated; absent packages remain explicit and no engine benchmark was claimed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Manifest complete | yes |
| Format / group | INT4 |
| Group size | 64 |
| ModelOpt handoff | no |
| TensorRT-LLM handoff | no |
| Numerical RMSE | 0.107446 |


## 7. Interpret rather than merely print

The manifest passed its required-field check and recorded scale SHA-256 `4fc993…d117e`. The numerical probe had RMSE 0.107446 and cosine 0.994265. Both ModelOpt and TensorRT-LLM handoff flags were false because the packages were unavailable.

That combination is a valid reproducibility artifact and an explicit stop. It supports preparing the handoff schema, not claims about FP8/INT4/FP4 engine quality or throughput.

**Inspection rule:** A valid manifest is a reproducibility result, not an engine throughput result.


## 8. Keep the evidence label honest

This run is labeled **`compatibility-probe`**. The named optional backend did not complete a native run in this environment. Package and failure evidence are retained; service or kernel performance is not inferred.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "The handoff contract was validated; absent packages remain explicit and no engine benchmark was claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "compatibility-probe",
  "executed_at_utc": "2026-08-07T14:45:52+00:00",
  "lesson": 17,
  "manifest": {
    "base_revision": "example-frozen-revision",
    "handoff": {
      "modelopt": false,
      "tensorrt_llm": false
    },
    "recipe": {
      "calibration": "synthetic-v1",
      "format": "INT4",
      "group_size": 64
    },
    "rollback_revision": "bf16-baseline-v1",
    "scale_sha256": "4fc993da767f7ef4e3cbfb4051a3448a622bc6955ec882016a7dd0ea0e5d117e"
  },
  "manifest_complete": true,
  "numerical_probe": {
    "cosine": 0.9942652,
    "mae": 0.0911599,
    "max_abs": 0.32194212,
    "rmse": 0.10744566
  },
  "schema_version": 1
}
S

## 9. Make the bounded decision

> Treat every tool boundary as a versioned artifact handoff with explicit validation and rollback metadata.

**Acceptance/rollback:** Validate a schema and hashes at each handoff, run a deterministic smoke sample, inspect engine layers, and keep quality and performance gates separate.

**Failure analysis:** Using `latest` model or container tags makes a manifest non-reproducible. Hashing scales but omitting the grouping axis can preserve bytes while changing meaning. Another failure is comparing engines built with different scheduler, tensor-parallel, or plugin settings and attributing the difference to quantization alone.


## 10. Extend the evidence

Run ModelOpt calibration in an isolated pinned container, export a checkpoint plus manifest, build a TensorRT-LLM engine, and add engine hash, builder flags, layer inspection, quality suite, and SLO report. Test that the rollback artifact loads under the same serving interface.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
